# 03 Tongue Field Trace

This notebook traces one Tongue field through properties, calibration, and hindcast outputs.

Full workflow context: [Tongue Data Assembly](../workflows/tongue-data-assembly.md) and [Tongue Calibration](../workflows/tongue-calibration.md)

Artifacts read here: the calibration container and the hindcast container.

In [1]:
from pathlib import Path

import pandas as pd

from swimrs.container import SwimContainer

ROOT = Path("/nas/swim/examples/tongue_ensemble")
CALIBRATION_CONTAINER = ROOT / "data" / "tongue_ensemble.swim"
HINDCAST_CONTAINER = ROOT / "data" / "tongue_hindcast.swim"
FIELD_UID = "1984"

calibration = SwimContainer.open(str(CALIBRATION_CONTAINER), mode="r")
hindcast = SwimContainer.open(str(HINDCAST_CONTAINER), mode="r")
FIELD_UID

'1984'

In [2]:
idx = hindcast.get_field_index(FIELD_UID)
pd.Series(
    {
        "awc": float(hindcast._root["properties/soils/awc"][idx]),
        "ksat": float(hindcast._root["properties/soils/ksat"][idx]),
        "irr": float(hindcast._root["properties/irrigation/irr"][idx]),
        "modis_lc": int(hindcast._root["properties/land_cover/modis_lc"][idx]),
    },
    name=FIELD_UID,
)

awc          0.151610
ksat        17.024981
irr          0.906825
modis_lc    12.000000
Name: 1984, dtype: float64

In [3]:
# Merged NDVI in the container is scene-sparse; drop gap days for a quick look.
hindcast.query.dataframe(
    "derived/merged_ndvi/inv_irr",
    fields=[FIELD_UID],
    start_date="2023-04-01",
    end_date="2023-06-15",
).dropna().head()

site,1984
time,
2023-04-03,0.260573
2023-04-08,0.254472
2023-04-25,0.317379
2023-04-28,0.366001
2023-04-30,0.396214


In [4]:
calibration_report = calibration.calibration_report()
calibration_report.to_dataframe().query("fid == @FIELD_UID")

,fid,aw,aw_std,kr_damp,kr_damp_std,ks_damp,ks_damp_std,mad,mad_std,ndvi_0,ndvi_0_std,ndvi_k,ndvi_k_std,swe_alpha,swe_alpha_std,swe_beta,swe_beta_std,irr,lulc
67,1984,243.849,52.292381,0.139748,0.051116,0.278649,0.165989,0.119631,0.013894,0.8,0.054846,8.83128,1.902262,0.286659,0.257976,1.35966,0.442663,0.906825,12


In [5]:
hindcast.run_dataframe(
    "hindcast_core",
    FIELD_UID,
    variables=["eta", "dperc", "swe", "irr_sim"],
    start_date="2023-05-01",
    end_date="2023-05-15",
).head()

,eta,dperc,swe,irr_sim
time,,,,
2023-05-01,4.152362,0.460477,0.0,4.604771
2023-05-02,4.975337,0.551892,0.0,5.518919
2023-05-03,4.401787,0.489715,0.0,4.897155
2023-05-04,5.256833,0.583149,0.0,5.831488
2023-05-05,4.394693,0.000000,0.0,0.000000
